# Recourse Fairness — baseline setup

Adult + COMPAS, baseline logistic regression, group fairness metrics before any mitigation.

In [ ]:
# pandas<3.0 — aif360 breaks on pandas 3's default string dtype
!pip install "pandas<3.0" -q
!pip install aif360 scikit-learn scipy matplotlib -q
print("installed — restart runtime now")

Restart runtime here (`Runtime` → `Restart session`), then continue. Don't rerun the cell above.

In [ ]:
import aif360, os

RAW = os.path.join(os.path.dirname(aif360.__file__), "data", "raw")

# COMPAS — ProPublica's original release
os.makedirs(f"{RAW}/compas", exist_ok=True)
!wget -q "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv" -O "{RAW}/compas/compas-scores-two-years.csv"

# Adult — raw UCI files, aif360 wants these three exact names
os.makedirs(f"{RAW}/adult", exist_ok=True)
!wget -q "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data" -O "{RAW}/adult/adult.data"
!wget -q "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test" -O "{RAW}/adult/adult.test"
!wget -q "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.names" -O "{RAW}/adult/adult.names"

for f in ["compas/compas-scores-two-years.csv", "adult/adult.data", "adult/adult.test", "adult/adult.names"]:
    p = f"{RAW}/{f}"
    size = os.path.getsize(p) if os.path.exists(p) else 0
    print(f, "OK" if size else "MISSING")

If Adult shows MISSING, UCI's moved the file again — send me whichever filename failed.

In [ ]:
from aif360.datasets import AdultDataset, CompasDataset

adult = AdultDataset()
compas = CompasDataset()

print("Adult:", adult.features.shape, adult.protected_attribute_names)
print("COMPAS:", compas.features.shape, compas.protected_attribute_names)

## Baseline

No mitigation yet — this is the "before" row for Table 1.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from aif360.metrics import ClassificationMetric

def run_baseline(dataset, protected_attr, priv, unpriv, name):
    train, test = dataset.split([0.8], shuffle=True, seed=42)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train.features)
    X_test = scaler.transform(test.features)

    clf = LogisticRegression(max_iter=2000)
    clf.fit(X_train, train.labels.ravel())

    test_pred = test.copy()
    test_pred.labels = clf.predict(X_test).reshape(-1, 1)

    metric = ClassificationMetric(
        test, test_pred,
        unprivileged_groups=[{protected_attr: unpriv}],
        privileged_groups=[{protected_attr: priv}],
    )

    print(f"--- {name} ({protected_attr}) ---")
    print("SPD:", metric.statistical_parity_difference())
    print("AOD:", metric.average_odds_difference())
    print("Acc:", metric.accuracy())
    print()
    return clf, scaler, test, test_pred, metric

adult_clf, adult_scaler, adult_test, adult_test_pred, adult_metric = run_baseline(
    adult, "sex", priv=1, unpriv=0, name="Adult"
)
compas_clf, compas_scaler, compas_test, compas_test_pred, compas_metric = run_baseline(
    compas, "race", priv=1, unpriv=0, name="COMPAS"
)

Reference run for COMPAS, yours won't match exactly (different split) but should be close:
SPD -0.176, AOD -0.154, Acc 0.672

Next: margin function, then Reweighing.

## Margin (recourse cost)

For every test point the model denies, this is its distance to the decision boundary in
scaled feature space — margin(x) = (w·x + b) / ‖w‖. Bigger distance, harder recourse.
Validated against sklearn's own `decision_function` before trusting any real numbers off it.

One change from the paper draft: computing this for every *unfavorably predicted* instance,
not "correctly classified and unfavorably predicted." Recourse is a property of what the
deployed model asks of you, it doesn't care whether the model's prediction happened to match
the true label — restricting to correct predictions would drop exactly the people (false
negatives) a fairness analysis should care most about. Methods section needs that sentence
fixed to match.

In [ ]:
def recourse_costs(clf, scaler, favorable_label, test_set, test_pred, protected_attr, priv, unpriv):
    X = scaler.transform(test_set.features)
    w = clf.coef_[0]
    b = clf.intercept_[0]
    w_norm = np.linalg.norm(w)

    margin = (X @ w + b) / w_norm
    assert np.allclose(margin, clf.decision_function(X) / w_norm), "margin doesn't match decision_function"

    predicted = test_pred.labels.ravel()
    denied = predicted != favorable_label
    cost = np.abs(margin[denied])

    attr_idx = test_set.protected_attribute_names.index(protected_attr)
    group = test_set.protected_attributes[:, attr_idx][denied]

    return cost[group == priv], cost[group == unpriv]


import numpy as np
from scipy.stats import mannwhitneyu, shapiro

def compare_recourse(priv_cost, unpriv_cost, name):
    print(f"--- {name} recourse cost ---")
    print(f"privileged   n={len(priv_cost)}  mean={priv_cost.mean():.4f}")
    print(f"unprivileged n={len(unpriv_cost)}  mean={unpriv_cost.mean():.4f}")

    # shapiro gets unreliable at large n, subsample if needed
    p_sample = priv_cost[:500]
    u_sample = unpriv_cost[:500]
    print(f"shapiro p (subsample): priv={shapiro(p_sample).pvalue:.4g}  unpriv={shapiro(u_sample).pvalue:.4g}")

    u_stat, p_val = mannwhitneyu(priv_cost, unpriv_cost)
    r = 1 - (2 * u_stat) / (len(priv_cost) * len(unpriv_cost))  # rank-biserial effect size
    print(f"Mann-Whitney U={u_stat:.1f}  p={p_val:.6f}  r={r:.4f}")
    print()

adult_priv_cost, adult_unpriv_cost = recourse_costs(
    adult_clf, adult_scaler, adult.favorable_label, adult_test, adult_test_pred, "sex", 1, 0
)
compas_priv_cost, compas_unpriv_cost = recourse_costs(
    compas_clf, compas_scaler, compas.favorable_label, compas_test, compas_test_pred, "race", 1, 0
)

compare_recourse(adult_priv_cost, adult_unpriv_cost, "Adult")
compare_recourse(compas_priv_cost, compas_unpriv_cost, "COMPAS")

Reference run, COMPAS (won't match exactly, different split, should be close):
n denied 476/1234, privileged mean 0.652, unprivileged mean 0.603, Mann-Whitney p = 0.854
— not significant on the unmitigated baseline. That's a real result, not a bug, worth sitting
with rather than expecting a dramatic gap before any mitigation has even been applied.

Copy all four numbers (both datasets, both groups, both p-values) into Table 2's Baseline row.
Next: Reweighing.

## Reweighing

First mitigation technique, pre-processing: reweights training rows by group/label combo
before the model ever sees them, doesn't touch features or labels, just how much attention
each row gets during fitting. Same margin and stats code from above, reused as-is on the new
model, that's the whole point of building it as functions earlier.

In [ ]:
from aif360.algorithms.preprocessing import Reweighing

def run_reweighing(dataset, protected_attr, priv, unpriv, name):
    train, test = dataset.split([0.8], shuffle=True, seed=42)

    RW = Reweighing(unprivileged_groups=[{protected_attr: unpriv}], privileged_groups=[{protected_attr: priv}])
    train_rw = RW.fit_transform(train)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train.features)
    X_test = scaler.transform(test.features)

    clf = LogisticRegression(max_iter=2000)
    clf.fit(X_train, train.labels.ravel(), sample_weight=train_rw.instance_weights)

    test_pred = test.copy()
    test_pred.labels = clf.predict(X_test).reshape(-1, 1)

    metric = ClassificationMetric(
        test, test_pred,
        unprivileged_groups=[{protected_attr: unpriv}],
        privileged_groups=[{protected_attr: priv}],
    )
    print(f"--- {name} after Reweighing ---")
    print("SPD:", metric.statistical_parity_difference())
    print("AOD:", metric.average_odds_difference())
    print("Acc:", metric.accuracy())

    return clf, scaler, test, test_pred


adult_rw_clf, adult_rw_scaler, adult_rw_test, adult_rw_pred = run_reweighing(
    adult, "sex", priv=1, unpriv=0, name="Adult"
)
compas_rw_clf, compas_rw_scaler, compas_rw_test, compas_rw_pred = run_reweighing(
    compas, "race", priv=1, unpriv=0, name="COMPAS"
)

adult_rw_priv, adult_rw_unpriv = recourse_costs(
    adult_rw_clf, adult_rw_scaler, adult.favorable_label, adult_rw_test, adult_rw_pred, "sex", 1, 0
)
compas_rw_priv, compas_rw_unpriv = recourse_costs(
    compas_rw_clf, compas_rw_scaler, compas.favorable_label, compas_rw_test, compas_rw_pred, "race", 1, 0
)

compare_recourse(adult_rw_priv, adult_rw_unpriv, "Adult, after Reweighing")
compare_recourse(compas_rw_priv, compas_rw_unpriv, "COMPAS, after Reweighing")

In [ ]:
from scipy.optimize import minimize

class FairLogisticRegression:
    def __init__(self, l2=1e-3, fair_lambda=5.0, max_iter=1000):
        self.l2, self.fair_lambda, self.max_iter = l2, fair_lambda, max_iter

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

    def _loss_and_grad(self, theta, X, y, s_c):
        w, b = theta[:-1], theta[-1]
        N = X.shape[0]
        scores = X @ w + b
        p = self._sigmoid(scores)
        eps = 1e-10
        bce = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
        C = np.mean(s_c * scores)
        total = bce + self.l2 * np.sum(w**2) + self.fair_lambda * C**2
        grad_w = X.T @ (p - y) / N + 2 * self.l2 * w + self.fair_lambda * 2 * C * (X.T @ s_c) / N
        grad_b = np.mean(p - y)
        return total, np.concatenate([grad_w, [grad_b]])

    def fit(self, X, y, s):
        s_c = s - s.mean()
        theta0 = np.zeros(X.shape[1] + 1)
        res = minimize(self._loss_and_grad, theta0, args=(X, y, s_c),
                        method="L-BFGS-B", jac=True, options={"maxiter": self.max_iter})
        self.coef_ = res.x[:-1].reshape(1, -1)
        self.intercept_ = np.array([res.x[-1]])
        self.converged_ = res.success
        return self

    def decision_function(self, X):
        return X @ self.coef_[0] + self.intercept_[0]

    def predict(self, X):
        return (self.decision_function(X) > 0).astype(int)


def run_fair_lr(dataset, protected_attr, priv, unpriv, name):
    train, test = dataset.split([0.8], shuffle=True, seed=42)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train.features)
    X_test = scaler.transform(test.features)

    attr_idx = train.protected_attribute_names.index(protected_attr)
    s_train = train.protected_attributes[:, attr_idx]

    clf = FairLogisticRegression(l2=1e-3, fair_lambda=5.0).fit(X_train, train.labels.ravel(), s_train)
    if not clf.converged_:
        print(f"WARNING: {name} optimizer did not converge, results below are not trustworthy")

    test_pred = test.copy()
    test_pred.labels = clf.predict(X_test).reshape(-1, 1)

    metric = ClassificationMetric(
        test, test_pred,
        unprivileged_groups=[{protected_attr: unpriv}],
        privileged_groups=[{protected_attr: priv}],
    )
    print(f"--- {name} after fairness-regularized LR ---")
    print("SPD:", metric.statistical_parity_difference())
    print("AOD:", metric.average_odds_difference())
    print("Acc:", metric.accuracy())

    return clf, scaler, test, test_pred


adult_fl_clf, adult_fl_scaler, adult_fl_test, adult_fl_pred = run_fair_lr(
    adult, "sex", priv=1, unpriv=0, name="Adult"
)
compas_fl_clf, compas_fl_scaler, compas_fl_test, compas_fl_pred = run_fair_lr(
    compas, "race", priv=1, unpriv=0, name="COMPAS"
)

adult_fl_priv, adult_fl_unpriv = recourse_costs(
    adult_fl_clf, adult_fl_scaler, adult.favorable_label, adult_fl_test, adult_fl_pred, "sex", 1, 0
)
compas_fl_priv, compas_fl_unpriv = recourse_costs(
    compas_fl_clf, compas_fl_scaler, compas.favorable_label, compas_fl_test, compas_fl_pred, "race", 1, 0
)

compare_recourse(adult_fl_priv, adult_fl_unpriv, "Adult, after fairness-regularized LR")
compare_recourse(compas_fl_priv, compas_fl_unpriv, "COMPAS, after fairness-regularized LR")

In [ ]:
import matplotlib.pyplot as plt

def effect_size(priv, unpriv):
    u, _ = mannwhitneyu(priv, unpriv)
    return 1 - (2 * u) / (len(priv) * len(unpriv))

methods = ["Baseline", "Reweighing", "Fair LR"]
adult_r = [
    effect_size(adult_priv_cost, adult_unpriv_cost),
    effect_size(adult_rw_priv, adult_rw_unpriv),
    effect_size(adult_fl_priv, adult_fl_unpriv),
]
compas_r = [
    effect_size(compas_priv_cost, compas_unpriv_cost),
    effect_size(compas_rw_priv, compas_rw_unpriv),
    effect_size(compas_fl_priv, compas_fl_unpriv),
]

x = np.arange(len(methods))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(x - width/2, adult_r, width, label="Adult")
ax.bar(x + width/2, compas_r, width, label="COMPAS")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.set_ylabel("Recourse effect size (rank-biserial r)")
ax.set_title("Recourse gap effect size by method and dataset")
ax.legend()
fig.tight_layout()
fig.savefig("figure1_effect_size.png", dpi=150)
plt.show()

print("Adult r:", [round(v, 4) for v in adult_r])
print("COMPAS r:", [round(v, 4) for v in compas_r])

In [ ]:
# Part 1: 30 random 80/20 splits x 3 methods x 2 datasets (takes roughly 10 min).
# "primary" = the exact setup used for every result so far.
# "sensitivity" = same, but the protected column is removed from the model inputs.
import os, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, norm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from aif360.algorithms.preprocessing import Reweighing

for name in ["adult", "compas", "recourse_costs", "FairLogisticRegression"]:
    assert name in globals(), f"'{name}' is missing, run the notebook top to bottom first (Runtime > Run all)"

OUT = "/content"
SEEDS = [42] + list(range(29))          # 42 = the split used so far
DATASETS = {"Adult": (adult, "sex"), "COMPAS": (compas, "race")}
METHODS = ["Baseline", "Reweighing", "Fair LR"]


def group_metrics(y_true, y_pred, s, fav):
    """SPD, AOD, accuracy. Same definitions as AIF360's ClassificationMetric (checked against seed 42 below);
    used because AIF360 caches every metric object it creates, which eats RAM over hundreds of fits."""
    hit, pos = y_pred == fav, y_true == fav
    u, p = s == 0, s == 1
    spd = hit[u].mean() - hit[p].mean()
    aod = 0.5 * ((hit[u & ~pos].mean() - hit[p & ~pos].mean()) + (hit[u & pos].mean() - hit[p & pos].mean()))
    return spd, aod, (y_pred == y_true).mean()


class Cols:
    """Lets recourse_costs see the same columns the model was trained on."""
    def __init__(self, scaler, keep):
        self.scaler, self.keep = scaler, keep

    def transform(self, F):
        return self.scaler.transform(F[:, self.keep])


def one_split(ds, attr, seed, method, drop_attr, lam=5.0):
    train, test = ds.split([0.8], shuffle=True, seed=seed)
    keep = np.array([i for i, n in enumerate(ds.feature_names) if not (drop_attr and n == attr)])
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(train.features[:, keep])
    Xte = scaler.transform(test.features[:, keep])
    ytr = train.labels.ravel()
    s_tr = train.protected_attributes[:, train.protected_attribute_names.index(attr)]

    converged = True
    if method == "Fair LR":
        clf = FairLogisticRegression(l2=1e-3, fair_lambda=lam).fit(Xtr, ytr, s_tr)
        converged = bool(clf.converged_)
    else:
        weights = None
        if method == "Reweighing":
            rw = Reweighing(unprivileged_groups=[{attr: 0}], privileged_groups=[{attr: 1}])
            weights = rw.fit_transform(train).instance_weights
        clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr, sample_weight=weights)

    pred = test.copy()
    pred.labels = clf.predict(Xte).reshape(-1, 1)
    grp = test.protected_attributes[:, test.protected_attribute_names.index(attr)]
    spd, aod, acc = group_metrics(test.labels.ravel(), pred.labels.ravel(), grp, ds.favorable_label)
    priv_cost, unpriv_cost = recourse_costs(clf, Cols(scaler, keep), ds.favorable_label, test, pred, attr, 1, 0)
    u, p = mannwhitneyu(priv_cost, unpriv_cost)

    # signed margin, flipped so that positive always means "toward the favorable decision"
    w = clf.coef_[0]
    sign = 1 if ds.favorable_label == 1 else -1
    margin = sign * clf.decision_function(Xte) / np.linalg.norm(w)
    names = [ds.feature_names[i] for i in keep]
    attr_w = sign * w[names.index(attr)] / np.linalg.norm(w) if attr in names else np.nan

    return dict(SPD=spd, AOD=aod, Acc=acc,
                n_priv=len(priv_cost), n_unpriv=len(unpriv_cost),
                cost_priv=priv_cost.mean(), cost_unpriv=unpriv_cost.mean(),
                r=1 - 2 * u / (len(priv_cost) * len(unpriv_cost)), p=p,
                attr_weight=attr_w,
                margin_priv=margin[grp == 1].mean(), margin_unpriv=margin[grp == 0].mean(),
                margin_sd_priv=margin[grp == 1].std(), margin_sd_unpriv=margin[grp == 0].std(),
                converged=converged)


rows, t0 = [], time.time()
for variant, drop in [("primary", False), ("sensitivity", True)]:
    for dname, (ds, attr) in DATASETS.items():
        for seed in SEEDS:
            for method in METHODS:
                rows.append(dict(variant=variant, dataset=dname, method=method, seed=seed,
                                 **one_split(ds, attr, seed, method, drop)))
        print(f"{variant:11s} {dname:6s} done  ({time.time() - t0:.0f}s elapsed)")
res = pd.DataFrame(rows)
res.to_csv(f"{OUT}/multiseed_results.csv", index=False)
print("non-converged fits:", int((~res.converged).sum()))

# seed 42 must reproduce the single-split table from before
ref = {("Adult", "Baseline"): (-0.1821, -0.0955, 0.4543), ("Adult", "Reweighing"): (-0.0781, 0.0745, 0.2460),
       ("Adult", "Fair LR"): (-0.0830, 0.0660, -0.0502), ("COMPAS", "Baseline"): (-0.1699, -0.1497, 0.0116),
       ("COMPAS", "Reweighing"): (0.0015, 0.0273, 0.0807), ("COMPAS", "Fair LR"): (-0.0636, -0.0381, 0.0762)}
bad = 0
for (d, m), (spd, aod, r) in ref.items():
    row = res[(res.variant == "primary") & (res.dataset == d) & (res.method == m) & (res.seed == 42)].iloc[0]
    if max(abs(row.SPD - spd), abs(row.AOD - aod), abs(row.r - r)) > 1e-3:
        bad += 1
        print("MISMATCH vs earlier table:", d, m)
print("seed-42 rows match the earlier table\n" if bad == 0 else f"{bad} mismatches, stop and send me this output\n")

# summary. the 0.1 cut-offs below are common rules of thumb, not a standard
res["group_fair"] = (res.SPD.abs() < 0.1) & (res.AOD.abs() < 0.1)
res["fair_and_gap"] = res.group_fair & (res.r.abs() >= 0.1)
g = res.groupby(["variant", "dataset", "method"], sort=False)
summary = pd.DataFrame({
    "SPD": g.SPD.mean(), "AOD": g.AOD.mean(), "Acc": g.Acc.mean(),
    "r_mean": g.r.mean(), "r_sd": g.r.std(),
    "r_p2.5": g.r.quantile(.025), "r_p97.5": g.r.quantile(.975),
    "share_r<0": g.r.apply(lambda x: (x < 0).mean()),
    "share_p<.05": g.p.apply(lambda x: (x < .05).mean()),
    "share_group_fair": g.group_fair.mean(),
    "share_fair_but_gap": g.fair_and_gap.mean(),
}).round(3)
summary.to_csv(f"{OUT}/multiseed_summary.csv")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
print(summary)

# same splits, Reweighing vs Fair LR: how different is group fairness, how different is recourse?
piv = res[res.variant == "primary"].pivot_table(index=["dataset", "seed"], columns="method", values=["SPD", "AOD", "r"])
dr = piv["r"]["Reweighing"] - piv["r"]["Fair LR"]
pair = pd.DataFrame({
    "max|dSPD|": (piv["SPD"]["Reweighing"] - piv["SPD"]["Fair LR"]).abs().groupby("dataset").max(),
    "max|dAOD|": (piv["AOD"]["Reweighing"] - piv["AOD"]["Fair LR"]).abs().groupby("dataset").max(),
    "dr_min": dr.groupby("dataset").min(), "dr_max": dr.groupby("dataset").max(),
}).round(3)
print("\nReweighing vs Fair LR on the same splits (primary):")
print(pair)

# where the mitigation acts. margins > 0 mean "toward the favorable decision"
mech = res[res.variant == "primary"].groupby(["dataset", "method"], sort=False)[
    ["attr_weight", "margin_priv", "margin_unpriv", "margin_sd_priv", "margin_sd_unpriv", "cost_priv", "cost_unpriv"]
].mean().round(3)
mech.to_csv(f"{OUT}/mechanism_table.csv")
print("\nMean over splits (primary). attr_weight = protected column's share of the weight vector, signed so + favors the privileged group:")
print(mech)

# figure
colors = {"Baseline": "#666666", "Reweighing": "#0072B2", "Fair LR": "#D55E00"}
fig, axes = plt.subplots(2, 2, figsize=(9.5, 7.2), sharey=True)
jit = np.random.default_rng(0)
for i, variant in enumerate(["primary", "sensitivity"]):
    for j, dname in enumerate(DATASETS):
        ax = axes[i, j]
        for k, method in enumerate(METHODS):
            d = res[(res.variant == variant) & (res.dataset == dname) & (res.method == method)]
            ax.scatter(k + jit.uniform(-0.13, 0.13, len(d)), d.r, s=14, alpha=0.55, color=colors[method], linewidths=0)
            lo, hi = d.r.quantile([.025, .975])
            ax.plot([k, k], [lo, hi], color="black", lw=1.3, zorder=2)
            ax.scatter(k, d.r.mean(), marker="_", s=700, color="black", zorder=3)
            ax.scatter(k + 0.22, d.loc[d.seed == 42, "r"].iloc[0], marker="D", s=26, facecolor="white",
                       edgecolor="black", zorder=4)
        ax.axhline(0, color="black", lw=0.8)
        ax.set_xticks(range(3))
        ax.set_xticklabels(METHODS)
        ax.set_title(f"{dname} ({'protected column is a model input' if variant == 'primary' else 'protected column removed'})",
                     fontsize=10)
        if j == 0:
            ax.set_ylabel("recourse gap, rank-biserial r\n(positive = unprivileged farther from boundary)", fontsize=9)
fig.suptitle("Recourse gap over 30 random splits\ndots = splits, vertical line = middle 95%, bar = mean, diamond = seed 42",
             fontsize=10)
fig.tight_layout()
fig.savefig(f"{OUT}/figure1_multiseed.png", dpi=200)
plt.show()

In [ ]:
# Part 2: penalty-strength sweep for Fair LR, COMPAS power check, library versions.
# Needs the previous cell to have run (reuses its functions and results).
import importlib.metadata as md

# Is the Adult reversal a knob or a fluke? Sweep fair_lambda (primary setup, first 10 splits).
LAMS = [0, 1, 2, 5, 10, 20, 50]
lam_rows, t1 = [], time.time()
for dname, (ds, attr) in DATASETS.items():
    for lam in LAMS:
        for seed in SEEDS[:10]:
            lam_rows.append(dict(dataset=dname, lam=lam, seed=seed,
                                 **one_split(ds, attr, seed, "Fair LR", False, lam=float(lam))))
    print(f"lambda sweep {dname:6s} done  ({time.time() - t1:.0f}s)")
lam_res = pd.DataFrame(lam_rows)
lam_res.to_csv(f"{OUT}/lambda_sweep.csv", index=False)
print("non-converged fits:", int((~lam_res.converged).sum()))
print(lam_res.groupby(["dataset", "lam"])[["SPD", "AOD", "Acc", "r"]].agg(["mean", "std"]).round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
x = np.arange(len(LAMS))
styles = {"SPD": ("#0072B2", "-", "SPD"), "AOD": ("#56B4E9", "--", "AOD"), "r": ("#D55E00", "-", "recourse gap r")}
for ax, dname in zip(axes, DATASETS):
    grp_l = lam_res[lam_res.dataset == dname].groupby("lam")
    for col, (color, ls, label) in styles.items():
        ax.errorbar(x, grp_l[col].mean(), yerr=grp_l[col].std(), color=color, ls=ls, marker="o", ms=4,
                    lw=2.4 if col == "r" else 1.6, capsize=2, label=label)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(LAMS)
    ax.set_xlabel("fair_lambda (penalty strength)")
    ax.set_title(f"{dname}: mean of 10 splits, bars = SD", fontsize=10)
axes[0].set_ylabel("metric value")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(f"{OUT}/figure3_lambda_sweep.png", dpi=200)
plt.show()

# power: what recourse gap could a COMPAS-sized sample have detected?
def power_mwu(r_true, n_priv, n_unpriv, reps=2000):
    rg = np.random.default_rng(0)
    shift = np.sqrt(2) * norm.ppf((1 + r_true) / 2)       # normal shift that gives this rank-biserial r
    hits = sum(mannwhitneyu(rg.normal(0, 1, n_priv), rg.normal(shift, 1, n_unpriv)).pvalue < .05 for _ in range(reps))
    return hits / reps

base = res[(res.variant == "primary") & (res.dataset == "COMPAS") & (res.method == "Baseline")]
n1, n2 = int(base.n_priv.median()), int(base.n_unpriv.median())
print(f"\nCOMPAS power (about {n1} vs {n2} denied, alpha = .05, normal-shift model):")
for r_true in [0.05, 0.10, 0.15, 0.20, 0.25]:
    print(f"  true r = {r_true:.2f}  ->  power = {power_mwu(r_true, n1, n2):.2f}")

# record the exact library versions used in this session
with open(f"{OUT}/requirements.txt", "w") as f:
    for pk in ["aif360", "pandas", "numpy", "scikit-learn", "scipy", "matplotlib"]:
        f.write(f"{pk}=={md.version(pk)}\n")
print("\n" + open(f"{OUT}/requirements.txt").read())

In [ ]:
import pandas as pd

def row(name, test, test_pred, protected_attr, priv, unpriv, priv_cost, unpriv_cost):
    m = ClassificationMetric(test, test_pred,
        unprivileged_groups=[{protected_attr: unpriv}], privileged_groups=[{protected_attr: priv}])
    u, p = mannwhitneyu(priv_cost, unpriv_cost)
    r = 1 - (2 * u) / (len(priv_cost) * len(unpriv_cost))
    return {
        "run": name,
        "SPD": round(m.statistical_parity_difference(), 4),
        "AOD": round(m.average_odds_difference(), 4),
        "Acc": round(m.accuracy(), 4),
        "recourse_priv_mean": round(priv_cost.mean(), 4),
        "recourse_unpriv_mean": round(unpriv_cost.mean(), 4),
        "mannwhitney_p": round(p, 6),
        "effect_size_r": round(r, 4),
    }

rows = [
    row("Adult - Baseline", adult_test, adult_test_pred, "sex", 1, 0, adult_priv_cost, adult_unpriv_cost),
    row("Adult - Reweighing", adult_rw_test, adult_rw_pred, "sex", 1, 0, adult_rw_priv, adult_rw_unpriv),
    row("Adult - Fair LR", adult_fl_test, adult_fl_pred, "sex", 1, 0, adult_fl_priv, adult_fl_unpriv),
    row("COMPAS - Baseline", compas_test, compas_test_pred, "race", 1, 0, compas_priv_cost, compas_unpriv_cost),
    row("COMPAS - Reweighing", compas_rw_test, compas_rw_pred, "race", 1, 0, compas_rw_priv, compas_rw_unpriv),
    row("COMPAS - Fair LR", compas_fl_test, compas_fl_pred, "race", 1, 0, compas_fl_priv, compas_fl_unpriv),
]

results_df = pd.DataFrame(rows)
results_df.to_csv("results_summary.csv", index=False)
results_df

In [ ]:
# Diagnostics for the Discussion (Adult, primary setup). Nothing existing is modified.
#   A. Reweighing vs Fair LR on the people BOTH models deny
#   B. margin-gap vs recourse-cost-gap relationship, and quantiles of the denied people's distances
# Outputs go to OUT/diagnostics/ (files named diag_*.csv). The original CSVs are only read (hash-checked before and after).
# Everything lives inside one function so no variable from the earlier cells is overwritten. Takes a few minutes.
import os, time, hashlib
import numpy as np, pandas as pd
from scipy.stats import mannwhitneyu
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from aif360.algorithms.preprocessing import Reweighing

for name in ["adult", "SEEDS", "OUT", "LAMS", "Cols", "recourse_costs", "FairLogisticRegression"]:
    assert name in globals(), f"'{name}' is missing, run cells 22 and 23 first (multi-seed and sweep cells)"


def run_diagnostics(TOL=1e-4):
    DIAG = f"{OUT}/diagnostics"
    os.makedirs(DIAG, exist_ok=True)
    ORIG = ["multiseed_results.csv", "multiseed_summary.csv", "mechanism_table.csv", "lambda_sweep.csv"]
    sha = lambda f: hashlib.sha256(open(f"{OUT}/{f}", "rb").read()).hexdigest()
    hash_before = {f: sha(f) for f in ORIG}
    ref_main = pd.read_csv(f"{OUT}/multiseed_results.csv")
    ref_main = ref_main[(ref_main.variant == "primary") & (ref_main.dataset == "Adult")].set_index(["method", "seed"])
    ref_sweep = pd.read_csv(f"{OUT}/lambda_sweep.csv")
    ref_sweep = ref_sweep[ref_sweep.dataset == "Adult"].set_index(["lam", "seed"])

    DS, ATTR = adult, "sex"
    FAV = DS.favorable_label
    SIGN = 1 if FAV == 1 else -1                # same convention as one_split(): margin > 0 = toward the favorable decision
    QS = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    METHODS3 = ["Baseline", "Reweighing", "Fair LR"]
    SWEEP_SEEDS = SEEDS[:10]                    # same 10 splits as the original sweep

    def rank_biserial(priv, unpriv):            # same formula as the notebook; r > 0: unprivileged farther
        u, p = mannwhitneyu(priv, unpriv)
        return 1 - 2 * u / (len(priv) * len(unpriv)), p

    def fit_split(seed, method, lam=5.0, direct_check=False):
        """Same split, scaling and fitting as one_split(..., drop_attr=False). Returns per-person arrays for the test set."""
        train, test = DS.split([0.8], shuffle=True, seed=seed)
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(train.features)
        Xte = scaler.transform(test.features)
        ytr = train.labels.ravel()
        s_tr = train.protected_attributes[:, train.protected_attribute_names.index(ATTR)]
        converged = True
        if method == "Fair LR":
            clf = FairLogisticRegression(l2=1e-3, fair_lambda=lam).fit(Xtr, ytr, s_tr)
            converged = bool(clf.converged_)
        else:
            weights = None
            if method == "Reweighing":
                rw = Reweighing(unprivileged_groups=[{ATTR: 0}], privileged_groups=[{ATTR: 1}])
                weights = rw.fit_transform(train).instance_weights
            clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr, sample_weight=weights)
        pred = clf.predict(Xte)
        denied = pred != FAV
        margin = SIGN * clf.decision_function(Xte) / np.linalg.norm(clf.coef_[0])
        grp = test.protected_attributes[:, test.protected_attribute_names.index(ATTR)]
        if direct_check:                        # the notebook's own recourse_costs() must return the same distances
            tp = test.copy()
            tp.labels = pred.reshape(-1, 1)
            pc, uc = recourse_costs(clf, Cols(scaler, np.arange(Xte.shape[1])), FAV, test, tp, ATTR, 1, 0)
            assert np.allclose(np.sort(pc), np.sort(np.abs(margin[denied & (grp == 1)]))) and \
                   np.allclose(np.sort(uc), np.sort(np.abs(margin[denied & (grp == 0)]))), "differs from recourse_costs()"
        return dict(margin=margin, denied=denied, grp=grp, converged=converged)

    def stats_of(f):
        m, d, g = f["margin"], f["denied"], f["grp"]
        cp, cu = np.abs(m[d & (g == 1)]), np.abs(m[d & (g == 0)])
        return dict(n_priv=len(cp), n_unpriv=len(cu), cost_priv=cp.mean(), cost_unpriv=cu.mean(),
                    margin_priv=m[g == 1].mean(), margin_unpriv=m[g == 0].mean(), r=rank_biserial(cp, cu)[0])

    diffs, n_cmp = [], [0]

    def compare(s, ref, tag):                   # recomputed numbers vs the original CSV row
        n_cmp[0] += 1
        for k in ["n_priv", "n_unpriv"]:
            if s[k] != ref[k]:
                diffs.append((tag, k, s[k], ref[k]))
        for k in ["cost_priv", "cost_unpriv", "margin_priv", "margin_unpriv", "r"]:
            if abs(s[k] - ref[k]) > TOL:
                diffs.append((tag, k, s[k], ref[k]))

    t_start = time.time()
    rows_cd, rows_q, obs, nonconv = [], [], [], 0
    for seed in SEEDS:
        F = {m: fit_split(seed, m, direct_check=(seed == 42)) for m in METHODS3}
        nonconv += sum(not F[m]["converged"] for m in METHODS3)
        g = F["Baseline"]["grp"]
        assert all(np.array_equal(F[m]["grp"], g) for m in METHODS3)
        for m in METHODS3:
            s = stats_of(F[m])
            compare(s, ref_main.loc[(m, seed)], f"primary {m} seed {seed}")
            obs.append(dict(set="primary", method=m, lam=5.0 if m == "Fair LR" else np.nan, seed=seed,
                            x=s["margin_priv"] - s["margin_unpriv"], y=s["cost_priv"] - s["cost_unpriv"], r=s["r"]))
        # A. people denied by both Reweighing and Fair LR
        both = F["Reweighing"]["denied"] & F["Fair LR"]["denied"]
        assert not (both & ~F["Reweighing"]["denied"]).any() and not (both & ~F["Fair LR"]["denied"]).any()
        row = dict(seed=seed, n_common_priv=int((both & (g == 1)).sum()), n_common_unpriv=int((both & (g == 0)).sum()))
        for tag, m in [("RW", "Reweighing"), ("FL", "Fair LR")]:
            d, mg = F[m]["denied"], F[m]["margin"]
            cp, cu = np.abs(mg[both & (g == 1)]), np.abs(mg[both & (g == 0)])
            r, p = rank_biserial(cp, cu)
            row.update({f"r_{tag}_common": r, f"p_{tag}_common": p,
                        f"cost_priv_{tag}_common": cp.mean(), f"cost_unpriv_{tag}_common": cu.mean(),
                        f"r_{tag}_own": stats_of(F[m])["r"],
                        f"n_{tag}_denied_priv": int((d & (g == 1)).sum()), f"n_{tag}_denied_unpriv": int((d & (g == 0)).sum()),
                        f"n_{tag}_only_priv": int((d & ~both & (g == 1)).sum()),
                        f"n_{tag}_only_unpriv": int((d & ~both & (g == 0)).sum())})
        row["dr_common"] = row["r_RW_common"] - row["r_FL_common"]
        row["dr_own"] = row["r_RW_own"] - row["r_FL_own"]
        rows_cd.append(row)
        # B2. quantiles of the denied people's distances to the boundary, per group and method
        for m in METHODS3:
            for gname, gv in [("privileged", 1), ("unprivileged", 0)]:
                c = np.abs(F[m]["margin"][F[m]["denied"] & (g == gv)])
                rows_q.append(dict(seed=seed, method=m, group=gname, n=len(c), mean=c.mean(), sd=c.std(),
                                   **{f"q{int(q * 100)}": np.quantile(c, q) for q in QS}))
    print(f"main loop done ({time.time() - t_start:.0f}s)")

    # B1. penalty sweep, refit from scratch (same 10 splits and strengths as the original sweep)
    for lam in LAMS:
        for seed in SWEEP_SEEDS:
            f = fit_split(seed, "Fair LR", lam=float(lam))
            nonconv += not f["converged"]
            s = stats_of(f)
            compare(s, ref_sweep.loc[(lam, seed)], f"sweep lam {lam} seed {seed}")
            obs.append(dict(set="sweep", method="Fair LR", lam=float(lam), seed=seed,
                            x=s["margin_priv"] - s["margin_unpriv"], y=s["cost_priv"] - s["cost_unpriv"], r=s["r"]))
    print(f"sweep done ({time.time() - t_start:.0f}s)")

    pd.set_option("display.width", 220)
    pd.set_option("display.max_columns", 40)
    print("\n" + "=" * 100 + "\nCHECKS")
    print(f"non-converged fits: {nonconv}")
    print(f"re-check against the original CSVs ({n_cmp[0]} fits, float tolerance {TOL}, counts exact): "
          + ("PASSED" if not diffs else f"FAILED, {len(diffs)} differences, first: {diffs[:3]}"))
    print("direct check against the notebook's recourse_costs() (seed 42, 3 methods): PASSED (it would have stopped otherwise)")

    # ------------------------------------------------------------ A. common-denied population
    cd = pd.DataFrame(rows_cd)
    cd["overlap_share_of_RW_denied"] = (cd.n_common_priv + cd.n_common_unpriv) / (cd.n_RW_denied_priv + cd.n_RW_denied_unpriv)
    cd["overlap_share_of_FL_denied"] = (cd.n_common_priv + cd.n_common_unpriv) / (cd.n_FL_denied_priv + cd.n_FL_denied_unpriv)
    cd.to_csv(f"{DIAG}/diag_common_denied_by_split.csv", index=False)
    cols = ["n_common_priv", "n_common_unpriv", "n_RW_only_priv", "n_RW_only_unpriv", "n_FL_only_priv", "n_FL_only_unpriv",
            "overlap_share_of_RW_denied", "overlap_share_of_FL_denied",
            "r_RW_own", "r_FL_own", "dr_own", "r_RW_common", "r_FL_common", "dr_common",
            "cost_priv_RW_common", "cost_unpriv_RW_common", "cost_priv_FL_common", "cost_unpriv_FL_common"]
    sa = cd[cols].agg(["mean", "min", "max"]).T.round(3)
    sa.to_csv(f"{DIAG}/diag_common_denied_summary.csv")
    print("\n" + "=" * 100 + "\nA. NEW RESULT: Reweighing vs Fair LR on the people denied by BOTH models (Adult, 30 splits)")
    print("   *_own = each model's own denied people (the existing result); *_common = the shared denied people (new)")
    print(sa)
    print(f"   dr_common > 0 on {(cd.dr_common > 0).sum()}/30 splits; r_FL_common < 0 on {(cd.r_FL_common < 0).sum()}/30; "
          f"r_RW_common > 0 on {(cd.r_RW_common > 0).sum()}/30; p_RW_common < .05 on {(cd.p_RW_common < .05).sum()}/30")

    # ------------------------------------------------------------ B1. mean-margin gap vs cost gap
    O = pd.DataFrame(obs)
    O.to_csv(f"{DIAG}/diag_mechanism_observations.csv", index=False)

    def ols(x, y):
        X = np.column_stack([np.ones_like(x), x])
        b = np.linalg.lstsq(X, y, rcond=None)[0]
        e = y - X @ b
        return b[0], b[1], 1 - (e ** 2).sum() / ((y - y.mean()) ** 2).sum(), np.sqrt((e ** 2).mean())

    def boot(d, B=2000):                        # resample whole splits (seeds) so rows of one split stay together
        rg = np.random.default_rng(0)
        parts = [(g_.x.values, g_.y.values) for _, g_ in d.groupby("seed")]
        out = np.empty((B, 4))
        for i in range(B):
            pick = rg.integers(0, len(parts), len(parts))
            a, b, r2, _ = ols(np.concatenate([parts[j][0] for j in pick]), np.concatenate([parts[j][1] for j in pick]))
            out[i] = a, b, r2, -a / b
        return np.percentile(out, [2.5, 97.5], axis=0)

    fit_rows = []
    print("\n" + "=" * 100 + "\nB1. NEW RESULT: cost_gap = a + b * margin_gap   (gap = privileged minus unprivileged)")
    print("   margin_gap = mean signed margin over ALL test people; cost_gap = mean distance over DENIED people")
    print(f"   sweep observations: strengths {list(LAMS)} x splits (seeds) {list(SWEEP_SEEDS)}")
    for name, d in [("sweep, all 7 penalty strengths x 10 splits", O[O.set == "sweep"]),
                    ("sweep, penalty >= 1 only", O[(O.set == "sweep") & (O.lam >= 1)]),
                    ("primary, 3 methods x 30 splits", O[O.set == "primary"])]:
        a, b, r2, rmse = ols(d.x.values, d.y.values)
        ci = boot(d)
        fit_rows.append(dict(observations=name, n=len(d), splits=d.seed.nunique(), intercept=a, slope=b, R2=r2, rmse=rmse,
                             intercept_lo=ci[0, 0], intercept_hi=ci[1, 0], slope_lo=ci[0, 1], slope_hi=ci[1, 1],
                             R2_lo=ci[0, 2], R2_hi=ci[1, 2], parity_margin_gap=-a / b, parity_lo=ci[0, 3], parity_hi=ci[1, 3]))
        print(f"   {name}: n={len(d)}  cost_gap = {a:+.3f} {b:+.3f} * margin_gap   R2 = {r2:.3f}  rmse = {rmse:.3f}")
        print(f"      95% bootstrap over splits: intercept [{ci[0,0]:+.3f}, {ci[1,0]:+.3f}], slope [{ci[0,1]:+.3f}, {ci[1,1]:+.3f}], "
              f"R2 [{ci[0,2]:.3f}, {ci[1,2]:.3f}], margin_gap at cost parity {-a / b:.3f} [{ci[0,3]:.3f}, {ci[1,3]:.3f}]")
    pd.DataFrame(fit_rows).round(4).to_csv(f"{DIAG}/diag_mechanism_fit.csv", index=False)

    # ------------------------------------------------------------ B2. quantiles among denied people
    q = pd.DataFrame(rows_q)
    q.to_csv(f"{DIAG}/diag_denied_quantiles_by_split.csv", index=False)
    stat_cols = ["mean", "q10", "q25", "q50", "q75", "q90", "q95", "q99"]
    summ = []
    for m in METHODS3:
        pv = q[(q.method == m) & (q.group == "privileged")].set_index("seed")[stat_cols]
        un = q[(q.method == m) & (q.group == "unprivileged")].set_index("seed")[stat_cols]
        gap = pv - un
        t = pd.DataFrame({"priv": pv.mean(), "unpriv": un.mean(), "gap_priv_minus_unpriv": gap.mean(),
                          "gap_min": gap.min(), "gap_max": gap.max(), "splits_gap>0": (gap > 0).sum()})
        t.insert(0, "method", m)
        summ.append(t.rename_axis("statistic").reset_index())
        print(f"\nB2. NEW RESULT: distance to the boundary among DENIED people, {m} (mean over 30 splits; gap>0 = privileged farther)")
        print(t.drop(columns="method").round(3))
    pd.concat(summ).round(4).to_csv(f"{DIAG}/diag_denied_quantiles_summary.csv", index=False)

    # ------------------------------------------------------------ integrity
    hash_after = {f: sha(f) for f in ORIG}
    print("\n" + "=" * 100)
    print("original CSVs unchanged:", hash_before == hash_after)
    assert hash_before == hash_after, "an original CSV changed, stop"
    print("new files in", DIAG, ":", sorted(os.listdir(DIAG)))


run_diagnostics()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)

axes[0].hist(adult_priv_cost, bins=25, alpha=0.6, label="privileged", density=True)
axes[0].hist(adult_unpriv_cost, bins=25, alpha=0.6, label="unprivileged", density=True)
axes[0].axvline(adult_priv_cost.mean(), color="C0", linestyle="--")
axes[0].axvline(adult_unpriv_cost.mean(), color="C1", linestyle="--")
axes[0].set_title("Baseline")
axes[0].set_xlabel("recourse cost (margin)")
axes[0].set_ylabel("density")
axes[0].legend()

axes[1].hist(adult_fl_priv, bins=25, alpha=0.6, label="privileged", density=True)
axes[1].hist(adult_fl_unpriv, bins=25, alpha=0.6, label="unprivileged", density=True)
axes[1].axvline(adult_fl_priv.mean(), color="C0", linestyle="--")
axes[1].axvline(adult_fl_unpriv.mean(), color="C1", linestyle="--")
axes[1].set_title("After fairness-regularized LR")
axes[1].set_xlabel("recourse cost (margin)")

fig.suptitle("Adult: recourse cost distribution by group, before and after")
fig.tight_layout()
fig.savefig("figure2_margin_distributions.png", dpi=150)
plt.show()

Reference, COMPAS after Reweighing: SPD 0.0015, AOD 0.027, Acc 0.649,
recourse priv mean 0.575, unpriv mean 0.625, Mann-Whitney p=0.154, r=0.081.
Group fairness improved sharply. Recourse gap did not, still not significant, same as baseline.
That's a real result: Reweighing fixed Question 1 without touching Question 2.

Copy all of this into Table 1 and Table 2's Reweighing row for both datasets.
Next: Prejudice Remover, same pattern again.